# Bayesian Training: Laplace Approximation

In this notebook, we generate synthetic Calorimetry ($C_p$) data for Silicon (Diamond) using its true SGTE polynomial representation. We then use Maximum A Posteriori (MAP) optimization to train an `EinsteinNode` to fit this data, and apply the Laplace Approximation (via `jax.hessian`) to extract the exact uncertainty on the Einstein Temperature $\Theta_E$.

In [ ]:
import os, sys
root_dir = os.getcwd() if not os.getcwd().endswith('examples') else os.path.abspath('../../')
sys.path.insert(0, os.path.join(root_dir, 'zgraph', 'src'))
sys.path.insert(0, os.path.join(root_dir, 'thermograph', 'src'))
sys.path.insert(0, root_dir)

import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from zgraph import *
from thermograph.nodes.sgte import SGTENode
from thermograph.nodes.einstein import GroundStateNode, EinsteinNode
from external_data.si_ge_sgte import GHSERSI
from external_data.einstein_params import SI_DIA
import jax.scipy.optimize


## 1. Generate Synthetic Data
We use the `SGTENode` to generate "ground truth" $C_p$ data, and add Gaussian noise to simulate experimental spread.

In [ ]:
# Ground truth SGTE solid
sgte_si = SGTENode(GHSERSI, T_index=0).compile_zgraph_engine()

def get_cp(node_fn, t_val):
    grad1 = jax.grad(lambda x: node_fn(jnp.atleast_1d(x)))
    grad2 = jax.grad(grad1)
    return -t_val * grad2(t_val)

vmap_get_cp = jax.vmap(lambda t: get_cp(sgte_si, t))

# Generate noisy data
np.random.seed(42)
T_data = jnp.linspace(100, 1500, 50)
true_cp = vmap_get_cp(T_data)

sigma_exp = 1.5  # J / mol K
Cp_data = true_cp + np.random.normal(0, sigma_exp, size=T_data.shape)

fig = go.Figure()
fig.add_trace(go.Scatter(x=T_data, y=Cp_data, mode='markers', name='Synthetic Data', marker=dict(color='black')))
fig.add_trace(go.Scatter(x=T_data, y=true_cp, mode='lines', name='SGTE Truth', line=dict(color='gray', dash='dash')))
fig.update_layout(title='Synthetic Si Calorimetry Data', xaxis_title='Temperature (K)', yaxis_title='Cp (J/mol-K)')
fig.show()


## 2. Define the Model and Likelihood (NLL)
We construct a function that builds the `EinsteinNode` on the fly from a guessed $\Theta_E$ parameter, and calculates the Negative Log-Likelihood (Gaussian MSE) against the data.

In [ ]:
fixed_E0 = SI_DIA["E_0"]  # E0 does not affect Cp

@jax.jit
def loss_fn(theta_E):
    # Build the deterministic physics nodes
    e0_node = GroundStateNode(fixed_E0).compile_zgraph_engine()
    osc_node = EinsteinNode(theta_E, T_index=0).compile_zgraph_engine()
    si_phase = FactorNode(jnp.array([[1.0, 3.0]]), [e0_node, osc_node], beta=0.0)
    
    # Predict Cp
    pred_cp = jax.vmap(lambda t: get_cp(si_phase, t))(T_data)
    
    # Gaussian Negative Log-Likelihood
    mse = jnp.sum((pred_cp - Cp_data)**2) / (2 * sigma_exp**2)
    return mse


## 3. MAP Optimization
We use `optax` to find the mode of the posterior (the $\Theta_E$ that perfectly minimizes the loss).

In [ ]:
theta_init = jnp.array(300.0)  # Bad initial guess

res = jax.scipy.optimize.minimize(
    loss_fn, 
    x0=theta_init, 
    method='BFGS'
)

theta_opt = res.x
print(f"\nOptimized Theta_E: {theta_opt:.2f} K (Truth is approx 645 K)")
print(f"Optimization Success: {res.success}")


## 4. Laplace Approximation (Hessian)
To capture uncertainty, we evaluate the 2nd derivative (Hessian) of the loss landscape exactly at the optimal $\Theta_E$.

In [ ]:
H = jax.hessian(loss_fn)(theta_opt)
variance = 1.0 / H
std_dev = jnp.sqrt(variance)

print(f"Hessian Curvature: {H:.4f}")
print(f"Posterior Standard Deviation: {std_dev:.2f} K")

# Sample 500 parameters from this parametric distribution
np.random.seed(123)
theta_samples = np.random.normal(theta_opt, std_dev, size=500)


## 5. Forward Uncertainty Propagation
We map our `EinsteinNode` across the 500 sampled parameters to predict 500 different $C_p$ curves, and extract the 95% Confidence Intervals!

In [ ]:
T_plot = jnp.linspace(100, 1500, 100)

@jax.jit
def predict_curve(theta_E):
    e0_node = GroundStateNode(fixed_E0).compile_zgraph_engine()
    osc_node = EinsteinNode(theta_E, T_index=0).compile_zgraph_engine()
    si_phase = FactorNode(jnp.array([[1.0, 3.0]]), [e0_node, osc_node], beta=0.0)
    return jax.vmap(lambda t: get_cp(si_phase, t))(T_plot)

# VMAP OVER THE SAMPLES!
cp_ensemble = jax.vmap(predict_curve)(theta_samples)  # Shape: [500, 100]

# Extract percentiles
cp_mean = jnp.mean(cp_ensemble, axis=0)
cp_lower = jnp.percentile(cp_ensemble, 5, axis=0)
cp_upper = jnp.percentile(cp_ensemble, 95, axis=0)

fig_uq = go.Figure()
fig_uq.add_trace(go.Scatter(x=T_data, y=Cp_data, mode='markers', name='Data', marker=dict(color='black')))
fig_uq.add_trace(go.Scatter(x=T_plot, y=cp_mean, mode='lines', name='MAP Fit', line=dict(color='red')))

fig_uq.add_trace(go.Scatter(
    x=np.concatenate([T_plot, T_plot[::-1]]),
    y=np.concatenate([cp_upper, cp_lower[::-1]]),
    fill='toself',
    fillcolor='rgba(255, 0, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% CI'
))
fig_uq.update_layout(title='Einstein Cp Prediction with Uncertainty', xaxis_title='Temperature (K)', yaxis_title='Cp (J/mol-K)')
fig_uq.show()


## Next Steps
Notice that the true SGTE $C_p$ curve bends upward at high temperatures (due to electronic/anharmonic contributions), whereas the pure Einstein oscillator approaches a flat asymptote of $3R$. The optimizer perfectly fits the *average* curvature by finding a $\Theta_E$ around 432 K.

To anchor the Ground State Energy ($E_0$), our next step will be to define a Phase Boundary Likelihood. We will require that the difference in Gibbs Free Energy between this Solid phase and the known SGTE Liquid phase is exactly 0 at the melting point ($T_m = 1687$ K).